## Topic: String output parser in LangChain

### 1. Introduction of String output parsers

- Definition:
    - StrOutputParser is the simplest and most commonly used output parser. It extracts the text content from the LLM's response message and returns it as a clean Python string.
    

- Problem
    - A chat model response is not always returned as a plain Python string.


- use class:
    - from langchain_core.output_parsers import StrOutputParser


### without string output parser
- Task (text-generation): 
    - 1. input 1: giving a topic(Dynamic) into LLM
    - 2. output 2 : generate the details report
    - 3. input 2: give details report into LLM
    - 4. output 2: generate the 5 line of summary from details report  

In [ ]:
### without string output parser
import os

from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate


# ------------------------------------------------------------------
# STEP 1: Load variables from the .env file
# ------------------------------------------------------------------
load_dotenv()


# ------------------------------------------------------------------
# STEP 2: Read the Hugging Face API token
# ------------------------------------------------------------------
hf_token = os.getenv("HF_TOKEN")


# ------------------------------------------------------------------
# STEP 3: Check whether the token was loaded
# ------------------------------------------------------------------
if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Please check your .env file."
    )


# ------------------------------------------------------------------
# STEP 4: Create the Hugging Face endpoint
# ------------------------------------------------------------------
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation",
    temperature=0.7,
    huggingfacehub_api_token=hf_token
)


# ------------------------------------------------------------------
# STEP 5: Create the ChatHuggingFace model
# ------------------------------------------------------------------
model = ChatHuggingFace(
    llm=llm
)


# ------------------------------------------------------------------
# STEP 6: Create the first prompt template
# ------------------------------------------------------------------
template1 = PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=["topic"]
)


# ------------------------------------------------------------------
# STEP 7: Create the second prompt template
# ------------------------------------------------------------------
template2 = PromptTemplate(
    template="Write a 5 line summary of the following text.\n{text}",
    input_variables=["text"]
)


# ------------------------------------------------------------------
# STEP 8: Fill the first prompt template
# ------------------------------------------------------------------
prompt1 = template1.invoke({
    "topic": "Data Science"
})


# ------------------------------------------------------------------
# STEP 9: Send the first prompt to the model
# ------------------------------------------------------------------
response_1 = model.invoke(prompt1)


# ------------------------------------------------------------------
# STEP 10: Create the second prompt
# ------------------------------------------------------------------
prompt2 = template2.invoke({
    "text": response_1.content
})


# ------------------------------------------------------------------
# STEP 11: Send the second prompt to the model
# ------------------------------------------------------------------
response_2 = model.invoke(prompt2)


# ------------------------------------------------------------------
# STEP 12: Display the final response
# ------------------------------------------------------------------
print(f"The 5 line summary:\n{response_2.content}")

### with string output parser

In [ ]:
### with string output parser with chain component
import os

from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ------------------------------------------------------------------
# STEP 1: Load variables from the .env file
# ------------------------------------------------------------------
load_dotenv()


# ------------------------------------------------------------------
# STEP 2: Read the Hugging Face API token
# ------------------------------------------------------------------
hf_token = os.getenv("HF_TOKEN")


# ------------------------------------------------------------------
# STEP 3: Check whether the token was loaded
# ------------------------------------------------------------------
if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Please check your .env file."
    )


# ------------------------------------------------------------------
# STEP 4: Create the Hugging Face endpoint
# ------------------------------------------------------------------
llm = HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation",
    temperature=0.7,
    huggingfacehub_api_token=hf_token
)


# ------------------------------------------------------------------
# STEP 5: Create the ChatHuggingFace model
# ------------------------------------------------------------------
model = ChatHuggingFace(
    llm=llm
)


# ------------------------------------------------------------------
# STEP 6: Create the first prompt template
# ------------------------------------------------------------------
template1 = PromptTemplate(
    template="Write a detailed report on {topic}",
    input_variables=["topic"]
)


# ------------------------------------------------------------------
# STEP 7: Create the second prompt template
# ------------------------------------------------------------------
template2 = PromptTemplate(
    template="Write a 5 line summary of the following text.\n{text}",
    input_variables=["text"]
)


# ------------------------------------------------------------------
# STEP 8: Create a string parser object
# ------------------------------------------------------------------
parser = StrOutputParser()

# ------------------------------------------------------------------
# STEP 9: Create a chain pipeline
# ------------------------------------------------------------------
# chain = prompt1 -> model -> parser (organize)-> prompt2 -> model -> parser (organize)
chain = template1 | model | parser | template2 | model | parser 

response = chain.invoke(
    {
        "topic": "Data Science"
    }
)


# ------------------------------------------------------------------
# STEP 12: Display the final response
# ------------------------------------------------------------------
print(response)

okay


In [ ]:
# Basic example 
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# ---------------------------------------------------------
# 1. Create the chat model
# ---------------------------------------------------------

model = ChatOpenAI(
    model="gpt-4o"
)


# ---------------------------------------------------------
# 2. Create the string output parser
# ---------------------------------------------------------

parser = StrOutputParser()


# ---------------------------------------------------------
# 3. Invoke the model
# ---------------------------------------------------------

response = model.invoke(
    "What is Artificial Intelligence?"
)


# ---------------------------------------------------------
# 4. Parse the model response into a string
# ---------------------------------------------------------

result = parser.invoke(response)


# ---------------------------------------------------------
# 5. Display the final string
# ---------------------------------------------------------

print(result)

In [ ]:
"""                               How It Works Internally

Step-by-Step Flow:
──────────────────

1. Prompt generates messages:
   [HumanMessage("Tell me a fun fact about Python.")]

2. LLM returns an AIMessage object:
   AIMessage(
       content="Python was named after Monty Python!",
       response_metadata={"token_usage": {"prompt_tokens": 12, ...}},
       id="run-abc123"
   )

3. StrOutputParser extracts ONLY the content:
   "Python was named after Monty Python!"
   
   It does this by calling: response.content
   And discarding everything else (metadata, IDs, etc.)


"""

In [ ]:
# Example 3
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# ---------------------------------------------------------
# 1. Create the prompt template
# ---------------------------------------------------------

prompt = ChatPromptTemplate.from_template(
    """
    Explain {topic} to a beginner
    in a simple and concise way.
    """
)


# ---------------------------------------------------------
# 2. Create the model
# ---------------------------------------------------------

model = ChatOpenAI(
    model="gpt-4o"
)


# ---------------------------------------------------------
# 3. Create the output parser
# ---------------------------------------------------------

parser = StrOutputParser()


# ---------------------------------------------------------
# 4. Build the chain
# ---------------------------------------------------------

chain = prompt | model | parser


# ---------------------------------------------------------
# 5. Invoke the chain
# ---------------------------------------------------------

result = chain.invoke({
    "topic": "Generative AI"
})


# ---------------------------------------------------------
# 6. Print the final output
# ---------------------------------------------------------

print(result)

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────┐
│              WHEN TO USE StrOutputParser                     │
│                                                              │
│  USE WHEN:                                                   │
│  ├── You need plain text output (summaries, translations)    │
│  ├── You're building a chatbot that displays text to users   │
│  ├── You're generating content (blogs, emails, poems)        │
│  ├── You're chaining multiple text transformations           │
│  ├── You want streaming output (typing effect)               │
│  └── You're prototyping and don't need structured data yet   │
│                                                              │
│  DON'T USE WHEN:                                             │
│  ├── You need to extract specific fields (name, age, etc.)   │
│  ├── You need to store data in a database with columns       │
│  ├── You need type safety (int, bool, list)                  │
│  ├── You're building an API that returns JSON                │
│  └── You need to make programmatic decisions based on output │
│      → Use JsonOutputParser or PydanticOutputParser instead  │
└──────────────────────────────────────────────────────────────┘

"""

- Key Takeaways:
   - StrOutputParser is a LangChain output parser that converts a model's output into a plain Python string.

   - It makes text generated by chat models easier to consume and compose within LangChain pipelines.

In [ ]:
### Two Core Responsibilities of Every Output Parser
"""
┌─────────────────────────────────────────────────────────┐
│           OUTPUT PARSER DOES TWO THINGS                 │
│                                                         │
│  1. FORMAT INSTRUCTIONS (Input Side)                    │
│     → Tells the LLM HOW to format its response          │
│     → Injected into the prompt automatically            │
│     → Example: "Return your answer as valid JSON."      │
│                                                         │
│  2. PARSING (Output Side)                               │
│     → Takes the LLM's raw text                          │
│     → Converts it into a Python object                  │
│     → Example: '{"name":"John"}' → {"name": "John"}     │
│                                                         │
│  parser.get_format_instructions()  → For the prompt     │
│  parser.parse(llm_output)          → For the response   │
└─────────────────────────────────────────────────────────┘


"""